# Phase 5 — Embedding V4: Two-Stage Contrastive Training

**Goal:** Train embedding v4 with augmented neutral data (4x neutral triplets) and sharper tau=0.07.

**Two-stage training:**
1. **Stage 1:** Train on augmented random triplets (`contrastive_triplets_aug.jsonl`, 2805 triplets)
2. **Hard negative mining:** Build FAISS index from Stage 1 → mine hard negatives
3. **Stage 2:** Train on hard triplets (`hard_contrastive_triplets_aug.jsonl`)
4. **Build FAISS index** from Stage 2 model (final)

**Key changes from v3:**
- tau: 0.12 → 0.07 (sharper distribution, larger margin)
- Neutral triplets: 100 → 400 (MAMS-ACSA augmentation)
- Two-stage hard negative mining

**Input:** `lcminhc/semeval-absa-restaurant` (XML + augmented data)

**Output:** Upload `/kaggle/working/outputs_p5_embed_v4/` as Kaggle dataset `p5-embed-v4`
- `embedding_v4_s1_best.pt` (Stage 1 checkpoint)
- `embedding_v4_s2_best.pt` (Stage 2 checkpoint — final)
- `train.faiss`, `metadata.json`, `train_vectors.npy` (FAISS index)
- Training logs

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml

In [ ]:
import os
import sys
import json
import shutil
import glob

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
# Wire dataset
KAGGLE_INPUT = None
for candidate in ['/kaggle/input/semeval-absa-restaurant',
                  '/kaggle/input/datasets/lcminhc/semeval-absa-restaurant']:
    if os.path.exists(candidate):
        KAGGLE_INPUT = candidate
        break
assert KAGGLE_INPUT, 'Dataset semeval-absa-restaurant not found'
print(f'Input: {KAGGLE_INPUT}')
print('Files:', os.listdir(KAGGLE_INPUT))

# Copy augmented triplets for embedding training
os.makedirs('data/processed', exist_ok=True)
for f in ['contrastive_triplets_aug.jsonl', 'classification_aug.jsonl',
          'sentiment_records_aug_300.jsonl']:
    src = f'{KAGGLE_INPUT}/{f}'
    dst = f'data/processed/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        with open(dst) as fp:
            n = sum(1 for _ in fp)
        print(f'{f}: {n} records')
    else:
        print(f'WARNING: {f} not found in dataset')

In [ ]:
import torch
import gc
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 1. Stage 1 — Train on Augmented Random Triplets

Config: `embedding_v4_s1.yaml`
- triplets: `contrastive_triplets_aug.jsonl` (2,805 — includes 400 neutral)
- tau=0.07, batch=128, epochs=15, patience=5
- gradient_checkpointing=true

In [ ]:
gc.collect()
torch.cuda.empty_cache()

!python scripts/02_train_embedding.py --config configs/embedding_v4_s1.yaml

In [ ]:
# Stage 1 training log
log_path = 'logs/embedding_v4_s1.jsonl'
print('=== Embedding V4 Stage 1 Training Log ===')
if os.path.exists(log_path):
    print(f'{"Ep":<4} {"Loss":<8} {"R@1":<6} {"R@3":<6} {"R@5":<6} {"intra":<7} {"inter":<7} {"ratio":<7} {"m1":<7} {"m2":<7}')
    print('-' * 68)
    with open(log_path) as f:
        for line in f:
            r = json.loads(line)
            print(f"{r['epoch']:<4} {r['train_loss']:<8.4f} "
                  f"{r.get('recall@1',0):<6.3f} {r.get('recall@3',0):<6.3f} {r.get('recall@5',0):<6.3f} "
                  f"{r.get('intra_sim',0):<7.4f} {r.get('inter_sim',0):<7.4f} {r.get('intra_inter_ratio',0):<7.3f} "
                  f"{r.get('margin_neg1',0):<7.3f} {r.get('margin_neg2',0):<7.3f}")
else:
    print('No log found.')

ckpt = 'checkpoints/embedding_v4_s1/best.pt'
if os.path.exists(ckpt):
    print(f'\nStage 1 checkpoint: {os.path.getsize(ckpt)/1e6:.1f} MB')
else:
    print('\nERROR: Stage 1 checkpoint not found!')

In [ ]:
# Save Stage 1 checkpoint immediately
output_dir = '/kaggle/working/outputs_p5_embed_v4'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f'{output_dir}/logs', exist_ok=True)

if os.path.exists('checkpoints/embedding_v4_s1/best.pt'):
    shutil.copy('checkpoints/embedding_v4_s1/best.pt',
                f'{output_dir}/embedding_v4_s1_best.pt')
    print('Stage 1 ckpt saved')
if os.path.exists('logs/embedding_v4_s1.jsonl'):
    shutil.copy('logs/embedding_v4_s1.jsonl',
                f'{output_dir}/logs/embedding_v4_s1.jsonl')
    print('Stage 1 log saved')

## 2. Hard Negative Mining

Use Stage 1 model to encode all train records → mine hard negatives.

In [ ]:
gc.collect()
torch.cuda.empty_cache()

!python scripts/build_hard_triplets.py \
    --embedding_ckpt checkpoints/embedding_v4_s1/best.pt \
    --cls_path data/processed/classification_aug.jsonl \
    --out_path data/processed/hard_contrastive_triplets_aug.jsonl

In [ ]:
# Verify hard triplets
hard_path = 'data/processed/hard_contrastive_triplets_aug.jsonl'
if os.path.exists(hard_path):
    with open(hard_path) as f:
        lines = f.readlines()
    print(f'Hard triplets: {len(lines)}')
    sample = json.loads(lines[0])
    print(f'Keys: {list(sample.keys())}')
    if 'pos_sim' in sample:
        print(f'Sample pos_sim={sample["pos_sim"]:.4f}, neg1_sim={sample["neg1_sim"]:.4f}')
else:
    print('ERROR: hard triplets not generated!')

## 3. Stage 2 — Train on Hard Triplets

Config: `embedding_v4_s2.yaml`
- triplets: `hard_contrastive_triplets_aug.jsonl`
- tau=0.07, batch=128, epochs=15, patience=5

In [ ]:
gc.collect()
torch.cuda.empty_cache()

!python scripts/02_train_embedding.py --config configs/embedding_v4_s2.yaml

In [ ]:
# Stage 2 training log
log_path = 'logs/embedding_v4_s2.jsonl'
print('=== Embedding V4 Stage 2 Training Log ===')
if os.path.exists(log_path):
    print(f'{"Ep":<4} {"Loss":<8} {"R@1":<6} {"R@3":<6} {"R@5":<6} {"intra":<7} {"inter":<7} {"ratio":<7} {"m1":<7} {"m2":<7}')
    print('-' * 68)
    with open(log_path) as f:
        for line in f:
            r = json.loads(line)
            print(f"{r['epoch']:<4} {r['train_loss']:<8.4f} "
                  f"{r.get('recall@1',0):<6.3f} {r.get('recall@3',0):<6.3f} {r.get('recall@5',0):<6.3f} "
                  f"{r.get('intra_sim',0):<7.4f} {r.get('inter_sim',0):<7.4f} {r.get('intra_inter_ratio',0):<7.3f} "
                  f"{r.get('margin_neg1',0):<7.3f} {r.get('margin_neg2',0):<7.3f}")
else:
    print('No log found.')

ckpt = 'checkpoints/embedding_v4_s2/best.pt'
if os.path.exists(ckpt):
    print(f'\nStage 2 checkpoint: {os.path.getsize(ckpt)/1e6:.1f} MB')
else:
    print('\nERROR: Stage 2 checkpoint not found!')

## 4. Build FAISS Index (from Stage 2 model)

Index sentiment_records_aug (augmented with neutral) for Phase 2a v2 retrieval.

In [ ]:
gc.collect()
torch.cuda.empty_cache()

!python scripts/03_build_index.py \
    --embedding_ckpt checkpoints/embedding_v4_s2/best.pt \
    --input data/processed/sentiment_records_aug_300.jsonl \
    --out_dir indexes/

In [ ]:
# Verify index
for f in ['train.faiss', 'metadata.json', 'train_vectors.npy']:
    path = f'indexes/{f}'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'{f}: {size_mb:.2f} MB')
    else:
        print(f'MISSING: {f}')

## 5. Embedding Quality Check

Quick sanity check: polarity match rate and cosine distribution.

In [ ]:
import numpy as np

vectors = np.load('indexes/train_vectors.npy')
with open('indexes/metadata.json') as f:
    metadata = json.load(f)

print(f'Vectors: {vectors.shape}')
print(f'Metadata: {len(metadata)} records')

# Random pair cosine similarity
rng = np.random.default_rng(42)
n = len(vectors)
idx = rng.integers(0, n, size=(500, 2))
random_sims = np.array([float(vectors[a] @ vectors[b]) for a, b in idx if a != b])
print(f'\nRandom pair cosine: mean={random_sims.mean():.4f}, std={random_sims.std():.4f}')
print(f'  min={random_sims.min():.4f}, max={random_sims.max():.4f}')

# Polarity match rate for top-k
import faiss
index = faiss.read_index('indexes/train.faiss')

k = 5
D, I = index.search(vectors, k + 1)  # +1 for self

match_counts = {pol: [] for pol in ['positive', 'negative', 'neutral']}
overall_matches = 0
overall_total = 0

for i in range(n):
    query_pol = metadata[i]['polarity']
    neighbors = [j for j in I[i] if j != i][:k]
    matches = sum(1 for j in neighbors if metadata[j]['polarity'] == query_pol)
    match_counts[query_pol].append(matches / k)
    overall_matches += matches
    overall_total += len(neighbors)

print(f'\nPolarity match rate (top-{k}):')
print(f'  Overall: {overall_matches/overall_total:.1%}')
for pol in ['positive', 'negative', 'neutral']:
    if match_counts[pol]:
        rates = np.array(match_counts[pol])
        print(f'  {pol}: {rates.mean():.1%} (n={len(rates)})')

## 6. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_p5_embed_v4'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f'{output_dir}/logs', exist_ok=True)

# Embedding checkpoints
for stage, ckpt_dir in [('s1', 'embedding_v4_s1'), ('s2', 'embedding_v4_s2')]:
    src = f'checkpoints/{ckpt_dir}/best.pt'
    if os.path.exists(src):
        shutil.copy(src, f'{output_dir}/embedding_v4_{stage}_best.pt')
        print(f'embedding_v4_{stage}_best.pt: {os.path.getsize(src)/1e6:.1f} MB')
    else:
        print(f'WARNING: {src} not found')

# Also save as embedding_best.pt (compatible name for downstream NB2/NB3)
s2_ckpt = 'checkpoints/embedding_v4_s2/best.pt'
if os.path.exists(s2_ckpt):
    shutil.copy(s2_ckpt, f'{output_dir}/embedding_best.pt')
    print('embedding_best.pt (= s2 final) saved')

# FAISS index
for f in ['train.faiss', 'metadata.json', 'train_vectors.npy']:
    src = f'indexes/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{output_dir}/{f}')
        print(f'{f}: {os.path.getsize(src)/1e6:.2f} MB')

# Hard triplets (for reference)
hard_path = 'data/processed/hard_contrastive_triplets_aug.jsonl'
if os.path.exists(hard_path):
    shutil.copy(hard_path, f'{output_dir}/hard_contrastive_triplets_aug.jsonl')
    print('hard_contrastive_triplets_aug.jsonl saved')

# Logs
for log in ['embedding_v4_s1.jsonl', 'embedding_v4_s2.jsonl']:
    src = f'logs/{log}'
    if os.path.exists(src):
        shutil.copy(src, f'{output_dir}/logs/{log}')

print(f'\n=== Output contents ===')
for root, dirs, files in os.walk(output_dir):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path) / 1e6
        rel = os.path.relpath(path, output_dir)
        print(f'  {rel}: {size:.2f} MB')

print(f'\nUpload as Kaggle dataset: p5-embed-v4')

In [ ]:
shutil.make_archive('/kaggle/working/outputs_p5_embed_v4_backup', 'zip',
                    '/kaggle/working', 'outputs_p5_embed_v4')
size_mb = os.path.getsize('/kaggle/working/outputs_p5_embed_v4_backup.zip') / 1e6
print(f'Backup zip: {size_mb:.1f} MB')